# Dam funnel: where removals drop out before the analysis

Of 2,126 listed removals, 530 reach the analysis table. This notebook explains the losses, to decide which are fixable and which are real data limits. It is **diagnostic only**: it reads the dam funnel audit and the existing pipeline outputs, and changes nothing.

1. **The funnel at a glance.** Dams lost at each stage, with counts and percentages.
2. **Rejected and ambiguous dam matches, by reason.**
3. **Listed dams with no matching record.** Why they never reached dam matching.
4. **Dams with samples on one side only.** Before-only vs after-only by removal decade: could a wider measurement window help?
5. **Potential payoff of the ambiguous dams** (separate estimate): how many before/after and BACI dams accepting the ambiguous dams' provisional river reaches could add.

Inputs: `data/processed/stage1_audit/<case_study>/dam_funnel/` (run first: `AUDIT=funnel sbatch data/slurm/stage1_audits.slurm`), the dam list, `dam_reaches/` and `reach_links/` outputs, the WBD watersheds, and the warehouse measurements (section 5).

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# Project root: the nearest folder at or above this one with etl/ and config/
# (works from any notebooks/ subfolder).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "etl").is_dir() and (p / "config").is_dir())
sys.path.insert(0, str(ROOT))

from analysis import mixed_model as mm
from analysis.audits.dam_funnel_audit import REASONS, REASON_ORDER
from etl.build_views import DEFAULT_MIN_SAMPLES, DEFAULT_MIN_YEARS
from etl.common.config import load_config
from etl.common.reach_inputs import dam_id_text, read_per_huc8
from etl.common.warehouse import warehouse_dir

DATA_ROOT = Path(os.environ.get("DAM_REMOVAL_PROJECT_ROOT", ROOT))
CASE_STUDY = "all_removals"
cfg = load_config(CASE_STUDY, project_root=DATA_ROOT)
FUNNEL_DIR = cfg.project_root / "data" / "processed" / "stage1_audit" / cfg.name / "dam_funnel"
WAREHOUSE = warehouse_dir(cfg)

window = cfg.get("wq_window", {}) or {}
YEARS_BEFORE, YEARS_AFTER = int(window.get("years_before", 5)), int(window.get("years_after", 5))
eligibility = cfg.get("eligibility", {}) or {}
MIN_SAMPLES = int(eligibility.get("min_samples", DEFAULT_MIN_SAMPLES))
MIN_YEARS = int(eligibility.get("min_years", DEFAULT_MIN_YEARS))
exposure = cfg.get("exposure", {}) or {}
MAX_DOWN, MAX_UP = exposure.get("max_downstream_km"), exposure.get("max_upstream_km")

dams = pd.read_csv(FUNNEL_DIR / "dam_funnel.csv", dtype={"AR_ID": str, "huc8": str}, low_memory=False)
# The dam list: removal years for dams that never reached dam_removal, and coordinates (section 3).
listed = pd.read_csv(cfg.path("dam_source"), dtype=str, low_memory=False)
listed["AR_ID"] = dam_id_text(listed["AR_ID"])
listed = listed.drop_duplicates("AR_ID")
listed_year = pd.to_numeric(listed.set_index("AR_ID")["Year_Removed"], errors="coerce")
listed_year = listed_year.where(listed_year > 0)
funnel = pd.read_csv(FUNNEL_DIR / "funnel_summary.csv")
MEANING = dict(REASONS)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 100); pd.set_option("display.float_format", lambda v: f"{v:,.1f}")

def pct(part, whole):
    """100 * part / whole; NaN where whole is 0 (numbers or columns)."""
    whole = pd.Series(whole, dtype=float).replace(0, np.nan) if np.ndim(whole) else (whole or np.nan)
    return 100 * part / whole

LISTED = len(dams)
print(f"{cfg.name}: {LISTED:,} listed dams; window -{YEARS_BEFORE}/+{YEARS_AFTER} years; "
      f"minimum {MIN_SAMPLES} samples in {MIN_YEARS} years per side; limits {MAX_DOWN} km down / {MAX_UP} km up")

## 1. The funnel at a glance

Each row is a stage of the pipeline, in order; a dam is counted at the **first** stage it fails. **% of listed** is out of all listed dams; **% of those reaching this stage** is the stage's own drop-out rate (a large share there means the stage itself is strict, a small one that few dams reach it).

In [ ]:
f = funnel.copy()
f["dams_reaching_stage"] = f["dams_left_after_this_stage"] + np.where(f["reason"].eq("in_analysis_table"), 0, f["dams"])
f["% of listed"] = pct(f["dams"], LISTED)
f["% of those reaching this stage"] = np.where(f["reason"].eq("in_analysis_table"), np.nan,
                                                pct(f["dams"], f["dams_reaching_stage"]))
display(f[["reason", "meaning", "dams_reaching_stage", "dams", "% of listed", "% of those reaching this stage",
           "dams_left_after_this_stage"]]
        .rename(columns={"dams": "dams lost here", "dams_reaching_stage": "dams reaching stage",
                         "dams_left_after_this_stage": "dams left"}))

lost = f[~f["reason"].eq("in_analysis_table")]
fig, ax = plt.subplots(figsize=(9, 4))
colors = ["#7f8c8d" if r.startswith(("not_matched", "match_", "accepted")) else
          "#c0392b" if r in ("no_samples_in_window", "one_side_only", "below_minimum") else "#2980b9"
          for r in lost["reason"]]
ax.barh(lost["meaning"][::-1], lost["dams"][::-1], color=colors[::-1])
for y, v in enumerate(lost["dams"][::-1]):
    ax.text(v, y, f" {v} ({pct(v, LISTED):.0f}%)", va="center", fontsize=8)
ax.set_xlabel("dams lost at this stage"); ax.set_title(
    f"{LISTED:,} listed → {int(f.loc[f['reason'].eq('in_analysis_table'), 'dams'].iloc[0])} in the analysis table   "
    "(grey: matching · blue: linking · red: samples)", fontsize=10)
ax.tick_params(axis="y", labelsize=8); fig.tight_layout(); plt.show()

## 2. Rejected and ambiguous dam matches, by reason

Dam matching places each dam on one NHDPlus river reach from evidence (reservoir outlet, river name, drainage area, distance). **Rejected** = no usable reach; **ambiguous** = a reach was found but the evidence didn't single it out, so the dam was set aside rather than guessed. **Dated** = has a removal year; undated dams couldn't be analysed even if matched. Plain-English meanings are below the table.

In [ ]:
REASON_TEXT = {
    "no_nhdplus_package": "no NHDPlus HR map published for its watershed (e.g. parts of Alaska)",
    "no_eligible_candidate": "no stream or river reach within the search distance",
    "datum_unsupported": "its coordinates use a datum that can't be converted",
    "sink_catchment_no_reach": "inside a closed basin with no river reach",
    "drainage_area_mismatch": "no nearby reach has a drainage area that fits",
    "confluence_ambiguous": "at a confluence: two different-sized rivers about equally close",
    "polygon_edge_ambiguous": "at the edge of a mapped water body, with a closer reach outside it",
    "far_without_support": "far from any reach, with no name or drainage-area support",
    "sink_catchment": "in a closed-basin catchment",
    "multiple_outlets": "its reservoir has several outlets of similar size",
    "reservoir_name_mismatch": "at a reservoir outlet, but the dam's river name matches neither the outlet nor the reach below",
}
m = dams[dams["reason"].isin(["match_rejected", "match_ambiguous"])].copy()
m["status"] = m["reason"].str.replace("match_", "")
m["dated"] = m["AR_ID"].map(listed_year).notna()
t = (m.groupby(["status", "match_reason"], dropna=False)
     .agg(dams=("AR_ID", "size"), dated=("dated", "sum")).reset_index())
t["% of status"] = t["dams"] / t.groupby("status")["dams"].transform("sum") * 100
t["% of listed"] = pct(t["dams"], LISTED)
t["meaning"] = t["match_reason"].map(REASON_TEXT).fillna("")
display(t.sort_values(["status", "dams"], ascending=[True, False]))
print(f"Rejected: {int((m.status == 'rejected').sum())}; ambiguous: {int((m.status == 'ambiguous').sum())}; "
      f"of these, dated: {int(m['dated'].sum())}")

fig, ax = plt.subplots(figsize=(9, 0.35 * len(t) + 1))
g = t.sort_values("dams")
ax.barh(g["status"] + ": " + g["match_reason"].fillna("?"), g["dams"],
        color=np.where(g["status"].eq("ambiguous"), "#e67e22", "#7f8c8d"))
ax.set_xlabel("dams"); ax.tick_params(axis="y", labelsize=8); ax.set_title("Dam-matching losses by reason")
fig.tight_layout(); plt.show()
if "state" in m:
    display(m.groupby(["state", "status"]).size().unstack(fill_value=0)
            .assign(total=lambda x: x.sum(axis=1)).sort_values("total", ascending=False).head(12))

## 3. Listed dams with no matching record

A dam reaches dam matching only if it has usable coordinates, is in the case study's dam list (`sampled_dams` in the config, which keeps dams whose point falls inside a watershed), and its watershed's matching output exists. Each dam is put in the **first** category that applies:

- **missing or invalid coordinates**: dropped when the dam list is loaded
- **outside every watershed polygon**: the point is offshore or outside the US watershed map, so it was never assigned a HUC8
- **watershed not in the case study**: its HUC8 isn't in the config's `huc8s`
- **inside a case-study watershed, but not in the config's dam list** (only when the config has `sampled_dams`)
- **watershed has no matching output** (the stage didn't produce a file for it)
- **in the watershed's matching output's area, but missing from it**

In [ ]:
from etl.common.config import PROJECT_ROOT_ENV_VAR  # noqa: F401  (documents the override)
import geopandas as gpd
from etl.common.wbd import resolve_huc8_for_points

none = listed[listed["AR_ID"].isin(dams.loc[dams["reason"].eq("not_matched_no_record"), "AR_ID"])].copy()
lat = pd.to_numeric(none["Latitude"], errors="coerce"); lon = pd.to_numeric(none["Longitude"], errors="coerce")
valid = lat.between(-90, 90) & lon.between(-180, 180)
sampled = {str(d["AR_ID"]): str(d.get("huc8", "")) for d in (cfg.get("sampled_dams") or [])}
reach_dir = cfg.project_root / "data" / "processed" / "dam_reaches" / cfg.name
huc_files = {p.stem.replace("dam_reaches_", "") for p in reach_dir.glob("dam_reaches_*.parquet")}

resolved = pd.Series(np.nan, index=none.index, dtype=object)
if valid.any():
    points = gpd.GeoDataFrame(none.loc[valid, ["AR_ID"]], geometry=gpd.points_from_xy(lon[valid], lat[valid]),
                              crs="EPSG:4326")
    joined = resolve_huc8_for_points(points, cfg.path("wbd_source"))
    resolved = joined["resolved_huc8"].groupby(level=0).first().reindex(none.index)
none["resolved_huc8"] = resolved
in_scope = none["resolved_huc8"].isin(set(cfg.huc8s))
# The watershed matching used: the config's dam list when it has one, else the resolved one.
none["huc8_used"] = none["AR_ID"].map(sampled).where(none["AR_ID"].isin(sampled), none["resolved_huc8"])
conditions = [~valid, valid & none["resolved_huc8"].isna(), none["resolved_huc8"].notna() & ~in_scope]
labels = ["missing or invalid coordinates", "outside every watershed polygon", "watershed not in the case study"]
if sampled:
    conditions.append(~none["AR_ID"].isin(sampled))
    labels.append("inside a case-study watershed, but not in the config's dam list")
conditions.append(~none["huc8_used"].isin(huc_files))
labels.append("watershed has no matching output")
none["category"] = np.select(conditions, labels, "in the watershed's matching output's area, but missing from it")
c = none["category"].value_counts().rename("dams").to_frame()
c["% of no-record dams"] = pct(c["dams"], len(none)); c["% of listed"] = pct(c["dams"], LISTED)
c["dated"] = none.assign(d=none["AR_ID"].map(listed_year).notna()).groupby("category")["d"].sum()
display(c)
print(f"{len(none)} no-record dams; resolved HUC8 in the case study: {int(in_scope.sum())}")
cols = [x for x in ["AR_ID", "Dam_Name", "State", "Latitude", "Longitude", "Year_Removed", "resolved_huc8"] if x in none]
none["Year_Removed"] = pd.to_numeric(none["Year_Removed"], errors="coerce").astype("Int64")
for category, group in none.groupby("category"):
    print(f"\n{category}: {len(group)} (first 5)")
    display(group[cols].head(5))
if "State" in none:
    display(none.groupby(["State", "category"]).size().unstack(fill_value=0)
            .assign(total=lambda x: x.sum(axis=1)).sort_values("total", ascending=False).head(10))

## 4. Dams with samples on one side only: would a wider window help?

These dams have samples within 5 years of removal, but never before **and** after at the same station for the same measure. Three kinds:

- **before only**: no samples after removal within 5 years
- **after only**: no samples before removal within 5 years
- **split**: samples on both sides, but at different stations or for different measures, so no single station × measure has both

**Post window incomplete** = removed so recently that 5 years after removal haven't passed yet; time, not a wider window, will fill those. A wider window could help **after-only** dams (if monitoring existed more than 5 years before removal) and **before-only dams with a complete post window** (if monitoring continued past 5 years). It can't help split dams. The warehouse only holds samples inside the ±5-year windows, so this shows where a wider window *could* help, not whether the older or later data exist; a small pilot download would answer that.

In [ ]:
one = dams[dams["reason"].eq("one_side_only")].copy()
one["kind"] = np.select([one["samples_pre"].gt(0) & one["samples_post"].eq(0),
                         one["samples_post"].gt(0) & one["samples_pre"].eq(0)],
                        ["before only", "after only"], "split")
one["incomplete"] = one["post_window_incomplete"].astype(str).str.lower().eq("true")
one["decade"] = (pd.to_numeric(one["removal_year"]) // 10 * 10).astype(int).astype(str) + "s"
one["wider window could help"] = np.select(
    [one["kind"].eq("after only"), one["kind"].eq("before only") & ~one["incomplete"],
     one["kind"].eq("before only") & one["incomplete"]],
    ["yes: earlier years", "yes: later years", "no: post window not over yet"], "no: split")
k = one["wider window could help"].value_counts().rename("dams").to_frame()
k["% of one-sided dams"] = pct(k["dams"], len(one)); k["% of listed"] = pct(k["dams"], LISTED)
display(k)
grid = pd.crosstab(one["decade"], one["kind"] + np.where(one["incomplete"], " (post window incomplete)", ""))
grid["total"] = grid.sum(axis=1)
display(grid)
plot = grid.drop(columns="total")
plot = plot.loc[plot.sum(axis=1) > 0]
ax = plot.plot.bar(stacked=True, figsize=(10, 3.8), rot=0,
                   color=[{"before only": "#2980b9", "after only": "#e67e22", "split": "#95a5a6"}.get(c.split(" (")[0], "#2980b9")
                          if "incomplete" not in c else "#aed6f1" for c in plot.columns])
ax.set_ylabel("dams"); ax.set_title(f"{len(one)} one-sided dams by removal decade"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

nos = dams[dams["reason"].eq("no_samples_in_window")]
outside = nos["samples_outside_window"].fillna(0).gt(0)
print(f"No samples in the window: {len(nos)} dams; {int(outside.sum())} of them have samples at their stations "
      f"outside the window (downloaded for another removal's window), a hint that data exist in other years.")

## 5. Potential payoff of the ambiguous dams (separate estimate)

**Question:** if the ambiguous dams were accepted on their provisional reach (after a review), how many would become usable for the models?

**How it is estimated.** Dam linking already connected each ambiguous dam to stations through its provisional reach; those links are kept but marked unusable. The same counting rules are applied to (a) the currently usable links (the **baseline**) and (b) the ambiguous dams' links, so the two are directly comparable:

1. stations whose own reach match was accepted, within the distance limits (30 km down / 10 km up)
2. samples within 5 years before / after removal (removal year left out), counted per station × measure, with the same minimum (3 samples in 2 years per side)
3. **before/after dam** = at least one downstream station meets the minimum; **strict** = that station has no closer known removal upstream (a rough stand-in for crediting each sample to its nearest removal)
4. **BACI dam** = a downstream **and** an upstream station meet the minimum for the same measure; upstream stations inside the mapped reservoir don't count as controls

**Bounds.** Measurements were downloaded only for stations in usable links, so a station linked *only* to an ambiguous dam has no data here: the estimate is a **lower bound**, and the number of such never-downloaded stations is shown as the unknown part. It is also optimistic in one way: a provisional reach may be wrong, which is why the dams need review before use.

In [ ]:
import duckdb

links = read_per_huc8(cfg, "reach_links", "reach_links", cfg.huc8s)
links["dam_id"] = dam_id_text(links["dam_id"]); links["station_id"] = links["station_id"].astype(str)
reaches = read_per_huc8(cfg, "dam_reaches", "dam_reaches", cfg.huc8s)
reaches["AR_ID"] = dam_id_text(reaches["AR_ID"])
year_of = pd.to_numeric(reaches.drop_duplicates("AR_ID").set_index("AR_ID")["Year_Removed"], errors="coerce")
year_of = year_of.where(year_of > 0)
ambiguous_ids = set(reaches.loc[reaches["match_status"].eq("ambiguous"), "AR_ID"])

links["side"] = np.where(links["relationship"].ne("same_reach"), links["relationship"],
                         np.where(links["station_fraction"] > links["dam_fraction"], "downstream",
                                  np.where(links["station_fraction"] < links["dam_fraction"], "upstream", None)))
km = links["network_distance_km"]
links["in_limits"] = (links["side"].eq("downstream") & (km <= MAX_DOWN if MAX_DOWN else True)) | \
                     (links["side"].eq("upstream") & (km <= MAX_UP if MAX_UP else True))
links["removal_year"] = links["dam_id"].map(year_of)
base = links[links["usable"].astype(bool) & links["in_limits"] & links["removal_year"].notna()]
amb = links[links["dam_id"].isin(ambiguous_ids) & links["station_status"].eq("accepted")
            & links["in_limits"] & links["removal_year"].notna()]

from analysis.audit_utils import series_sql
con = duckdb.connect()
downloaded = set(con.sql(f"SELECT station_id FROM read_parquet('{(WAREHOUSE / 'wq_station.parquet').resolve()}')")
                 .df()["station_id"].astype(str))
con.execute(f"""CREATE VIEW meas AS SELECT station_id, year(sample_date) AS year,
                {series_sql()} AS series
                FROM read_parquet('{(WAREHOUSE / 'wq_measurement.parquet').resolve()}')""")
print(f"Usable links in limits (baseline): {len(base):,} ({base['dam_id'].nunique()} dams); ambiguous dams' links "
      f"to accepted stations in limits: {len(amb):,} ({amb['dam_id'].nunique()} dams)")

# Nearest known removal per downstream station (usable links of dated dams), for the strict count.
nearest_km = base[base["side"].eq("downstream")].groupby("station_id")["network_distance_km"].min()

def payoff(pairs):
    """Per dam: before/after (any / strict) and BACI eligibility under the current rules."""
    pairs = pairs[["dam_id", "station_id", "side", "network_distance_km", "station_in_mapped_reservoir",
                   "removal_year"]].assign(station_in_mapped_reservoir=lambda f: f["station_in_mapped_reservoir"]
                                           .fillna(False).astype(bool))
    con.register("pairs", pairs)
    g = con.sql(f"""
        SELECT p.dam_id, p.station_id, p.side, m.series, p.network_distance_km, p.station_in_mapped_reservoir,
               count(*) FILTER (WHERE m.year < p.removal_year) AS n_pre,
               count(*) FILTER (WHERE m.year > p.removal_year) AS n_post,
               count(DISTINCT m.year) FILTER (WHERE m.year < p.removal_year) AS y_pre,
               count(DISTINCT m.year) FILTER (WHERE m.year > p.removal_year) AS y_post
        FROM pairs p JOIN meas m USING (station_id)
        WHERE m.year - p.removal_year BETWEEN -{YEARS_BEFORE} AND {YEARS_AFTER} AND m.year <> p.removal_year
        GROUP BY ALL""").df()
    con.unregister("pairs")
    g["has_both"] = (g["n_pre"] >= MIN_SAMPLES) & (g["y_pre"] >= MIN_YEARS) & (g["n_post"] >= MIN_SAMPLES) & (g["y_post"] >= MIN_YEARS)
    ok = g[g["has_both"]]
    down = ok[ok["side"].eq("downstream")]
    other = down["station_id"].map(nearest_km)
    strict = down[other.isna() | (down["network_distance_km"] <= other)]
    up = ok[ok["side"].eq("upstream") & ~ok["station_in_mapped_reservoir"]]
    baci = down.merge(up[["dam_id", "series"]].drop_duplicates(), on=["dam_id", "series"])
    baci_strict = strict.merge(up[["dam_id", "series"]].drop_duplicates(), on=["dam_id", "series"])
    return {"before/after (any)": set(down["dam_id"]), "before/after (strict)": set(strict["dam_id"]),
            "BACI (any)": set(baci["dam_id"]), "BACI (strict)": set(baci_strict["dam_id"])}, baci

base_sets, _ = payoff(base)
amb_sets, amb_baci = payoff(amb)
summary = pd.DataFrame({"current (usable links)": {k: len(v) for k, v in base_sets.items()},
                        "added by ambiguous dams": {k: len(amb_sets[k] - base_sets[k]) for k in base_sets}})
summary["% gain"] = pct(summary["added by ambiguous dams"], summary["current (usable links)"])
display(summary)
print(f"Check: the baseline before/after (any) count should be close to the funnel's dams with a downstream "
      f"has_both station ({int(dams['has_both_downstream'].astype(str).str.lower().eq('true').sum())}); "
      "it uses the same window and minimum, but these simplified rules, which is why the comparison above is "
      "made with the same rules on both sides.")

never = amb[~amb["station_id"].isin(downloaded)]
print(f"Unknown part: {never['station_id'].nunique()} stations linked only to ambiguous dams were never downloaded "
      f"({never['dam_id'].nunique()} dams; {never.loc[never['side'].eq('downstream'), 'dam_id'].nunique()} with such a "
      "downstream station). Their potential isn't counted above.")
print("BACI by measure (ambiguous dams, any):")
display(amb_baci.groupby("series")["dam_id"].nunique().rename("dams").sort_values(ascending=False).to_frame())

**Which ambiguous dams would pay off, by matching reason.** The reasons tell you how hard the review is: a **reservoir name mismatch** is usually quick to check on a map; **confluence** and **far without support** cases need a closer look at which river the dam was on. Review the dams that would add BACI comparisons first.

In [ ]:
amb_dams = reaches[reaches["AR_ID"].isin(ambiguous_ids)].drop_duplicates("AR_ID").set_index("AR_ID")
amb_dams["dated"] = amb_dams.index.map(year_of).notna()
amb_dams["any in-limit station"] = amb_dams.index.isin(set(amb["dam_id"]))
for key in ("before/after (any)", "before/after (strict)", "BACI (any)", "BACI (strict)"):
    amb_dams[key] = amb_dams.index.isin(amb_sets[key] - base_sets[key])
by_reason = amb_dams.groupby("match_reason")[["dated", "any in-limit station", "before/after (any)",
                                               "before/after (strict)", "BACI (any)", "BACI (strict)"]].sum()
by_reason.insert(0, "ambiguous dams", amb_dams.groupby("match_reason").size())
display(by_reason.astype(int).sort_values("ambiguous dams", ascending=False))

review = amb_dams[amb_dams["before/after (any)"]].copy()
review["Year_Removed"] = pd.to_numeric(review["Year_Removed"], errors="coerce").astype("Int64")
review["BACI measures"] = review.index.map(amb_baci.groupby("dam_id")["series"].nunique()).fillna(0).astype(int)
cols = [c for c in ["Dam_Name", "River", "huc8", "match_reason", "Year_Removed", "BACI measures",
                    "BACI (strict)", "before/after (strict)"] if c in review.columns]
print(f"{len(review)} ambiguous dams would add a before/after dam; review order: most BACI measures first")
display(review.sort_values(["BACI measures", "BACI (strict)", "before/after (strict)"], ascending=False)[cols].head(30))

## Reading the results and next steps

- **Stages in grey (matching)** are partly fixable by review; section 5 says how much a review of the ambiguous dams would pay off, and section 3 whether the no-record dams are a data issue (coordinates, outside the watershed map) or a pipeline gap (missing output).
- **Stages in red (samples)** are mostly real data limits. Section 4 separates what a wider window could plausibly add from what only time will fix (recent removals).
- Every number here is a **count of candidates**, not a result: nothing becomes part of the analysis without a reviewed change, and any change should be decided before looking at its effect on the estimates.